# 03. Transiciones electrónicas y espectros

Este cuaderno utiliza el método `transition` para calcular las propiedades de los fotones emitidos o absorbidos durante transiciones entre niveles de energía.

## Objetivos

- Distinguir emisión y absorción.
- Calcular la energía del fotón, su frecuencia y su longitud de onda.
- Construir una tabla de transiciones.
- Explorar algunas líneas de las series espectrales del hidrógeno.


In [ ]:
import matplotlib.pyplot as plt

from bohr_atom import BohrAtom, PROTON_MASS_KG

hydrogen = BohrAtom(
    Z=1,
    nucleus_mass_kg=PROTON_MASS_KG,
    name="Hidrógeno"
)


## 1. Emisión y absorción

Una transición descendente (`n_i > n_f`) corresponde a emisión. Una transición ascendente (`n_i < n_f`) corresponde a absorción.

La variación de energía del átomo tiene signo opuesto al intercambio de energía del fotón:

\[
\Delta E_{atomo}=E_f-E_i, \qquad E_\gamma=|\Delta E_{atomo}|.
\]


In [ ]:
emission = hydrogen.transition(3, 2)
absorption = hydrogen.transition(2, 3)

print("TRANSICIÓN 3 → 2")
print("Proceso:", emission["process"])
print(f"Cambio de energía del átomo: {emission['delta_energy_atom_j']:.6e} J")
print(f"Energía del fotón: {emission['photon_energy_ev']:.6f} eV")
print(f"Longitud de onda: {emission['wavelength_nm']:.3f} nm")

print("\nTRANSICIÓN 2 → 3")
print("Proceso:", absorption["process"])
print(f"Cambio de energía del átomo: {absorption['delta_energy_atom_j']:.6e} J")
print(f"Energía del fotón: {absorption['photon_energy_ev']:.6f} eV")
print(f"Longitud de onda: {absorption['wavelength_nm']:.3f} nm")


## 2. Tabla de transiciones

Calcularemos algunas transiciones de emisión hacia los niveles `n_f=1`, `n_f=2` y `n_f=3`.


In [ ]:
transitions_to_analyze = [
    (2, 1),
    (3, 1),
    (4, 1),
    (3, 2),
    (4, 2),
    (5, 2),
    (4, 3),
    (5, 3),
]

results = []

for n_initial, n_final in transitions_to_analyze:
    result = hydrogen.transition(n_initial, n_final)
    results.append(result)

print(
    f"{'Transición':>12} {'Proceso':>12} "
    f"{'E fotón (eV)':>16} {'Frecuencia (Hz)':>20} "
    f"{'Longitud (nm)':>16}"
)
print("-" * 82)

for result in results:
    transition_label = (
        f"{result['initial_level']} → {result['final_level']}"
    )
    print(
        f"{transition_label:>12} "
        f"{result['process']:>12} "
        f"{result['photon_energy_ev']:>16.6f} "
        f"{result['frequency_hz']:>20.6e} "
        f"{result['wavelength_nm']:>16.3f}"
    )


## 3. Serie de Balmer

La serie de Balmer agrupa transiciones de emisión que terminan en `n_f=2`, partiendo de niveles superiores:

\[
n_i=3,4,5,\ldots \quad\longrightarrow\quad n_f=2.
\]

Calcularemos las primeras cinco líneas dentro del modelo implementado.


In [ ]:
balmer_results = []

for n_initial in range(3, 8):
    result = hydrogen.transition(n_initial, 2)
    balmer_results.append(result)

print(f"{'Línea':>10} {'Transición':>14} {'Longitud de onda (nm)':>24}")
print("-" * 52)

for index, result in enumerate(balmer_results, start=1):
    print(
        f"H{index}: {('' + str(result['initial_level']) + ' → 2'):>10} "
        f"{result['wavelength_nm']:>24.3f}"
    )


## 4. Representación de las líneas de Balmer

El siguiente gráfico es una representación tipo espectro de líneas. La posición horizontal corresponde a la longitud de onda calculada; no representa intensidades experimentales.


In [ ]:
wavelengths = [item["wavelength_nm"] for item in balmer_results]
labels = [
f"{item['initial_level']}→2" for item in balmer_results]

fig, ax = plt.subplots(figsize=(10, 3.5))

for wavelength, label in zip(wavelengths, labels):
    ax.axvline(wavelength, linewidth=2)
    ax.text(
        wavelength,
        0.05,
        label,
        rotation=90,
        ha="right",
        va="bottom"
    )

ax.set_xlim(min(wavelengths) - 15, max(wavelengths) + 15)
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_xlabel("Longitud de onda (nm)")
ax.set_title("Líneas de la serie de Balmer calculadas con el modelo de Bohr")
ax.grid(axis="x", alpha=0.25)
plt.show()


## 5. Comprobar las relaciones del fotón

Para cada transición deben cumplirse:

\[
E_\gamma=h
u, \qquad \lambda=rac{c}{
u}.
\]

La implementación devuelve la energía, la frecuencia y la longitud de onda calculadas a partir de las constantes físicas utilizadas por el paquete.


In [ ]:
result = hydrogen.transition(3, 2)

h = hydrogen.planck_constant
c = hydrogen.speed_of_light

energy_from_frequency = h * result["frequency_hz"]
wavelength_from_frequency = c / result["frequency_hz"]

print(
    "Consistencia Eγ = hν:",
    abs(result["photon_energy_j"] - energy_from_frequency) < 1e-30
)

print(
    "Consistencia λ = c/ν:",
    abs(result["wavelength_m"] - wavelength_from_frequency) < 1e-20
)


## Interpretación y limitaciones

- Las longitudes de onda son predicciones del modelo no relativista de Bohr con corrección por masa reducida.
- No se incluyen estructura fina, desplazamiento de Lamb ni otras correcciones de espectroscopía de alta precisión.
- El gráfico de líneas muestra posiciones espectrales, no intensidades ni anchuras de línea.

**Siguiente cuaderno:** diagramas de niveles y representación simplificada de órbitas.
